# Istio Ambient, end to end — a petshop walkthrough

**Solo Enterprise for Istio, ambient mode, on kind.** One app (a petshop), every ambient security feature, and the Solo Enterprise differences called out as we go.

In [ ]:
# Run from the lab directory. Constants used throughout.
[ -d istio-ambient-cert-identity-kind ] && cd istio-ambient-cert-identity-kind || :
export CTX=kind-cert-identity
export ISTIO_NS=istio-system
# license (edit if your secrets live elsewhere)
export SECRETS_FILE="${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}"
[ -f "$SECRETS_FILE" ] && set -a && . "$SECRETS_FILE" && set +a
echo "context: $CTX ; license set: $([ -n "$SOLO_ISTIO_LICENSE_KEY" ] && echo yes || echo NO)"


## 1 · Stand up the mesh

Ambient here is the **four Solo-distribution Helm charts** (`base`, `istiod`, `cni`, `ztunnel`), profile `ambient`


In [ ]:
# 1a. kind cluster (1 control-plane + 1 worker)
kind create cluster --config kind/cluster.yaml


**Kick off the Solo UI now, in the background.** The Gloo Platform management plane (mgmt server, agent, UI, Prometheus, Redis, OTel) has no Istio dependency and takes the longest of anything in §1 — so start it the moment the cluster exists and let it install while we stand up ambient. §2 checks the result and port-forwards. `gloo-platform` 2.13.x pairs with Istio 1.30.


In [ ]:
# 1b. Solo UI (Gloo Platform mgmt plane) — full install, backgrounded to a log.
export GLOO_MESH_NS=gloo-mesh GLOO_PLATFORM_VERSION=2.13.2
(
  helm repo add gloo-platform https://storage.googleapis.com/gloo-platform/helm-charts || true
  helm repo update gloo-platform
  kubectl --context $CTX create namespace $GLOO_MESH_NS --dry-run=client -o yaml | kubectl --context $CTX apply -f -

  helm --kube-context $CTX upgrade -i gloo-platform-crds gloo-platform/gloo-platform-crds \
    -n $GLOO_MESH_NS --version $GLOO_PLATFORM_VERSION --wait

  # register the cluster BEFORE the main install (the CRD is in place now)
  kubectl --context $CTX apply -f - <<'REG'
apiVersion: admin.gloo.solo.io/v2
kind: KubernetesCluster
metadata: { name: cert-identity, namespace: gloo-mesh }
spec: { clusterDomain: cluster.local }
REG

  helm --kube-context $CTX upgrade -i gloo-platform gloo-platform/gloo-platform \
    -n $GLOO_MESH_NS --version $GLOO_PLATFORM_VERSION -f - <<VALUES
common: { cluster: cert-identity }
licensing:
  glooMeshLicenseKey: "${GLOO_PLATFORM_LICENSE_KEY:-$SOLO_ISTIO_LICENSE_KEY}"
glooMgmtServer: { enabled: true, createGlobalWorkspace: true }
glooUi: { enabled: true, serviceType: ClusterIP }
glooAgent:
  enabled: true
  relay: { serverAddress: gloo-mesh-mgmt-server.gloo-mesh:9900 }
prometheus: { enabled: true }
redis: { deployment: { enabled: true } }
telemetryCollector: { enabled: true }
telemetryGateway: { enabled: true }
glooInsightsEngine: { enabled: true }
VALUES
  echo "GLOO INSTALL DONE"
) > /tmp/cert-identity-gloo-install.log 2>&1 &
echo "Solo UI installing in the background — log: /tmp/cert-identity-gloo-install.log"


In [ ]:
# 1c. Gateway API CRDs (used by the waypoint later)
kubectl --context $CTX apply --server-side -f \
  https://github.com/kubernetes-sigs/gateway-api/releases/download/v1.5.1/standard-install.yaml


In [ ]:
# 1d. Pre-pull the Solo Istio images on the host and load them into kind.
#     (docker save | kind load, because kind's containerd chokes on multi-arch indexes.)
REG=us-docker.pkg.dev/soloio-img/istio ; VER=1.30.3-solo
case "$(uname -m)" in arm64|aarch64) PLAT=linux/arm64;; *) PLAT=linux/amd64;; esac
for img in pilot proxyv2 install-cni ztunnel; do
  docker pull -q $REG/$img:$VER
  docker save --platform $PLAT $REG/$img:$VER -o /tmp/$img.tar
  kind load image-archive /tmp/$img.tar --name cert-identity && rm -f /tmp/$img.tar
done


On the 1.30 line the chart version **and** the image tag both keep the `-solo` suffix (`1.30.3-solo`). Keep it on the tag — the plain `1.30.3` image in the same registry is the upstream build, with none of the Solo additions. The trust domain is set directly with `meshConfig.trustDomain: cert-identity`, so identities are `spiffe://cert-identity/ns/<ns>/sa/<sa>` — **not** `cluster.local`.


In [ ]:
# helm repo + versions for the Solo distribution
export HUB=us-docker.pkg.dev/soloio-img/istio TAG=1.30.3-solo
export HREPO=oci://us-docker.pkg.dev/soloio-img/istio-helm HVER=1.30.3-solo

# 1e. base — Istio CRDs + cluster roles
helm --kube-context $CTX upgrade -i istio-base $HREPO/base \
  -n $ISTIO_NS --create-namespace --version $HVER --set defaultRevision=default --wait


In [ ]:
# 1f. istiod (profile ambient) — LICENCE, TRUST DOMAIN and access logs are VALUES, not patches
helm --kube-context $CTX upgrade -i istiod $HREPO/istiod -n $ISTIO_NS --version $HVER --wait -f - <<EOF
profile: ambient
global: { hub: $HUB, tag: $TAG }
istio_cni: { enabled: true }
license: { value: $SOLO_ISTIO_LICENSE_KEY }
env: { PILOT_SKIP_VALIDATE_TRUST_DOMAIN: "true" }
meshConfig:
  accessLogFile: /dev/stdout
  trustDomain: cert-identity
EOF


In [ ]:
# 1g. istio-cni (traffic capture) + ztunnel (per-node L4 proxy).
#     ztunnel gets JSON logs + Solo L7 telemetry as env VALUES — again, no kubectl patch.
helm --kube-context $CTX upgrade -i istio-cni $HREPO/cni -n $ISTIO_NS --version $HVER --wait -f - <<EOF
profile: ambient
global: { hub: $HUB, tag: $TAG }
ambient: { dnsCapture: true }
excludeNamespaces: [istio-system, kube-system]
EOF

helm --kube-context $CTX upgrade -i ztunnel $HREPO/ztunnel -n $ISTIO_NS --version $HVER --wait -f - <<EOF
profile: ambient
hub: $HUB
tag: $TAG
namespace: $ISTIO_NS
istioNamespace: $ISTIO_NS
env: { LOG_FORMAT: json, L7_ENABLED: "true" }
EOF
kubectl --context $CTX -n $ISTIO_NS rollout status daemonset/ztunnel daemonset/istio-cni-node --timeout=180s
echo "ambient ready (Helm, no operator), trust domain cert-identity, JSON access logs on"


**Kick off the IdP now, in the background.** Keycloak takes ~40–60s to deploy.


In [ ]:
kubectl --context $CTX apply -f yaml/40-idp/keycloak.yaml
echo "Keycloak starting in the background — carry on; it'll be up by the JWT step"


## 2 · Watch it in the Solo UI (Gloo UI)

The **Gloo UI** is Solo's own dashboard for Solo Enterprise for Istio. It is the Gloo Platform management plane; on one kind cluster the mgmt server, the agent and the UI are co-located, and once the cluster is registered the agent discovers the ambient mesh. It has been **installing in the background since §1** — check it finished, then port-forward.


In [ ]:
# 2a. the plane has been installing since §1 — confirm the background install,
#     then wait only for the UI (the one thing we port-forward)
for i in $(seq 1 60); do grep -q "GLOO INSTALL DONE" /tmp/cert-identity-gloo-install.log && break; sleep 5; done
grep -q "GLOO INSTALL DONE" /tmp/cert-identity-gloo-install.log \
  || { echo "background install has not finished — last log lines:"; tail -5 /tmp/cert-identity-gloo-install.log; }
kubectl --context $CTX -n gloo-mesh rollout status deploy/gloo-mesh-ui --timeout=300s


Port-forward the UI in the **background**


In [ ]:
# 2b. background port-forward on a FREE random local port (8090 is often taken
#     by another lab). nohup + & returns straight away; logs to /tmp.
PORT=$(python3 -c 'import socket;s=socket.socket();s.bind(("",0));print(s.getsockname()[1]);s.close()')
nohup kubectl --context $CTX -n gloo-mesh port-forward svc/gloo-mesh-ui $PORT:8090 \
  > /tmp/cert-identity-gloo-ui-pf.log 2>&1 &
sleep 4
echo "Gloo UI → http://localhost:$PORT"


Open the URL the previous cell printed. Deploy the petshop in the next step and refresh: `petstore`, `storefront`, `analytics` and both `checkout` workloads appear under the `petshop` namespace, and once traffic flows the Observability graph fills in.


## 3 · The petshop app

One namespace, enrolled in ambient with a single label. The cast:

| Workload | ServiceAccount → identity | Role |
|---|---|---|
| `petstore` | `sa/petstore` | the API (`GET /pets`, `DELETE /pets/{id}`) |
| `storefront` | `sa/storefront` | client the L4 policy **allows** |
| `analytics` | `sa/analytics` | client the L4 policy **denies** |
| `checkout-blue` | `sa/checkout` | shares one SA with green … |
| `checkout-green` | `sa/checkout` | … so both present the **same** SVID |

**Where the traffic comes from:** each client runs a `curl http://petstore:8080/pets` loop *every 2 seconds* and prints the HTTP code. The observe-cells below don't send requests themselves — they read the client's last log line (`kubectl logs --tail=1`) to see the current outcome, which is why each policy step does a short `sleep` first (to let the new rule reach ztunnel). A denied client shows `000000` (connection refused), an allowed one `200`.


In [ ]:
# Enrol the namespace in ambient — one label, no restarts.
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: v1
kind: Namespace
metadata:
  name: petshop
  labels:
    istio.io/dataplane-mode: ambient
EOF

# Deploy the petshop (manifests in yaml/10-app/ — petstore is the API, the rest are clients)
kubectl --context $CTX apply -f yaml/10-app/
kubectl --context $CTX -n petshop rollout status deploy/petstore deploy/storefront deploy/analytics deploy/checkout-blue deploy/checkout-green --timeout=180s
kubectl --context $CTX -n petshop get pods


## 4 · Identity is the certificate

ztunnel holds one mTLS SVID per workload **identity**, and the identity is the ServiceAccount — the URI SAN is exactly what an L4 policy matches on. So look for what is *missing* below: five pods, but only **four** leaf certs. `checkout-blue` and `checkout-green` never appear by name; they share `sa/checkout`, so ztunnel presents **one** cert for both. That is the ceiling §7 hits, and §13 removes.


In [ ]:
# five pods, but their identity is the ServiceAccount…
kubectl --context $CTX -n petshop get pods \
  -o custom-columns=POD:.metadata.name,SERVICEACCOUNT:.spec.serviceAccountName

# …so the ztunnel on their node holds only FOUR leaf certs — checkout appears once
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=cert-identity-worker -o jsonpath='{.items[0].metadata.name}')
istioctl --context $CTX ztunnel-config certificate "$ZT.$ISTIO_NS" | grep -E "CERTIFICATE NAME|petshop" | grep -Ei "name|leaf"


## 5 · Authorize on identity, at L4

One `AuthorizationPolicy`, enforced by ztunnel, no waypoint. It selects `petstore` and allows only the `storefront` identity — so it simultaneously **allows storefront** and **denies analytics and checkout** (ztunnel fails closed: once any ALLOW selects a workload, everything unnamed is denied).

Principals use the trust domain `cert-identity`


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: allow-storefront, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - from: [{ source: { principals: ["cert-identity/ns/petshop/sa/storefront"] } }]
    to:   [{ operation: { ports: ["8080"] } }]
EOF
sleep 10
# each client curls petstore every 2s and prints the HTTP code
for d in storefront analytics checkout-blue checkout-green; do
  echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"
done


**Checked the Gloo UI Graph and the denied workloads still show edges to `petstore`?** The graph is drawn from **completed-request telemetry** over a rolling window (15 minutes by default) — those edges are the last 15 minutes of *pre-policy* 200s still ageing out, not the denied attempts. A workload that is denied at L4 never completes a request, so once its history rolls out of the window it draws nothing (§8's warehouse client demonstrates this: denied from its first packet, it never appears at all). The graph shows **what the mesh serves**; the per-connection **allow/deny verdicts** are in the ztunnel access logs — the next section.

**Make the graph react faster for a demo:** the telemetry itself is quick — the collector scrapes ztunnel every 15s and the UI refreshes every 15s, so fresh data is on screen within ~30 seconds. What lingers is history: use the **time-interval picker in the Graph toolbar** (next to Refresh) and drop it to the smallest window — the graph then reshapes within a minute or two of a policy change.


## 6 · Read the L4 access logs

ztunnel logs every connection with the peer SPIFFE identities and the outcome — identity-aware audit at L4, no waypoint. The clients are calling `petstore` continuously, so there are always fresh lines; with §5's `allow-storefront` policy in force you'll see `storefront` **ALLOW** and `analytics` **DENY**, each tagged with its `src.identity`.


In [ ]:
# ztunnel's L4 access logs, projected to who -> what + the outcome.
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=cert-identity-worker -o jsonpath='{.items[0].metadata.name}')
kubectl --context $CTX -n $ISTIO_NS logs "$ZT" --tail=400 \
 | grep '"scope":"access"' | grep petshop | tail -8 \
 | jq -rc '{src:(.["src.identity"]//"-"),
            dst:(.["dst.service"]//.["dst.identity"]//"-"),
            dir:(.direction//"-"),
            result:(if (.error//"")=="" then "ALLOW" else ("DENY: "+(.error|tostring)) end)}'


## 7 · The shared-ServiceAccount ceiling

Add `sa/checkout` to the allowed set. Both `checkout-blue` and `checkout-green` get in — and there is **no L4 rule that lets one in and keeps the other out**, because they present the same SVID. This is the limit of SA-scoped identity, and the motivation for §13.


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: allow-checkout, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - from: [{ source: { principals: ["cert-identity/ns/petshop/sa/checkout"] } }]
    to:   [{ operation: { ports: ["8080"] } }]
EOF
sleep 10
for d in storefront analytics checkout-blue checkout-green; do
  echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"
done
# -> storefront 200, checkout-blue 200, checkout-green 200, analytics still 000000


## 8 · More of the L4 match surface — namespace, `when`, and DENY

Every policy so far matched on **identity** (the SPIFFE principal). ztunnel can decide on more than that, still at L4, still with no waypoint. The things it can see about a connection:

- **who** is calling — `principals` (the identity), which is what we've used so far
- **where** the caller is — its source `namespaces` (or source IP)
- **where it's going** — the destination `port`

Two more moves round it out: any of these can be written as a CEL **`when`** clause (not only as a top-level `from`/`to` field), and a **`DENY`** rule always beats an `ALLOW`.

To show a decision based on **where** a caller is (rather than who it is), we add a second client, `warehouse-svc`, in a *different* namespace (`warehouse`), then allow only callers whose namespace is `petshop` — so `warehouse-svc` is refused.


In [ ]:
# A second client, in ANOTHER namespace, so we can decide on WHERE a caller is.
# It has its own identity (spiffe://cert-identity/ns/warehouse/sa/warehouse-svc)
# and calls petstore across namespaces. (Same manifest as yaml/25-l4-surface/00-warehouse.yaml.)
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: v1
kind: Namespace
metadata:
  name: warehouse
  labels:
    istio.io/dataplane-mode: ambient
---
apiVersion: v1
kind: ServiceAccount
metadata: { name: warehouse-svc, namespace: warehouse }
---
apiVersion: apps/v1
kind: Deployment
metadata: { name: warehouse-svc, namespace: warehouse, labels: { app: warehouse-svc } }
spec:
  replicas: 1
  selector: { matchLabels: { app: warehouse-svc } }
  template:
    metadata: { labels: { app: warehouse-svc } }
    spec:
      serviceAccountName: warehouse-svc
      containers:
        - name: client
          image: curlimages/curl:8.14.1
          command: ["/bin/sh","-c"]
          args:
            - |
              while true; do
                code=$(curl -s -o /dev/null -w '%{http_code}' --max-time 3 http://petstore.petshop:8080/pets || echo 000)
                echo "$(date -u +%H:%M:%S) warehouse-svc -> GET petstore.petshop/pets : $code"
                sleep 2
              done
EOF
kubectl --context $CTX -n warehouse rollout status deploy/warehouse-svc --timeout=90s


**Watching in the Gloo UI?** Add `warehouse` to the Graph's namespace picker — and don't expect a node. The graph draws from completed-request telemetry, and `warehouse-svc` is denied at L4 from its very first packet, so it never completes a request and never appears. That absence is the policy working. The graph shows what the mesh **serves**; what it **refuses** is in the ztunnel access logs (§6) — look for DENY lines with `src.identity: spiffe://cert-identity/ns/warehouse/sa/warehouse-svc`.


**Decide on the source namespace — with a CEL `when` clause.** Allow only callers whose `source.namespace` is `petshop`, so the cross-namespace `warehouse-svc` is denied. (`from.source.namespaces: ["petshop"]` is the equivalent without a `when`; the `when` form generalises to any L4 attribute ztunnel knows.)


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-allow-petshop-namespace, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - to:   [{ operation: { ports: ["8080"] } }]
    when: [{ key: source.namespace, values: ["petshop"] }]
EOF
sleep 14
echo "petshop:   $(kubectl --context $CTX -n petshop   logs deploy/storefront    --tail=1)"
echo "warehouse: $(kubectl --context $CTX -n warehouse logs deploy/warehouse-svc --tail=1)"


**DENY beats ALLOW.** ztunnel evaluates CUSTOM → DENY → ALLOW, so a `DENY` rule overrides the namespace `ALLOW`. Here `analytics` is in `petshop` (the ALLOW would admit it) but a `DENY` on its identity blocks it anyway — a clean carve-out with no change to the broader policy.


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-deny-analytics, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: DENY
  rules:
  - from: [{ source: { principals: ["cert-identity/ns/petshop/sa/analytics"] } }]
EOF
sleep 14
for d in storefront analytics; do echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"; done
# storefront 200 (still allowed), analytics 000000 (denied: DENY > ALLOW)

# reset before the waypoint section
kubectl --context $CTX -n petshop delete authorizationpolicy l4-allow-petshop-namespace l4-deny-analytics --ignore-not-found


## 9 · Add a waypoint for L7

Everything so far was L4. HTTP concerns — methods, paths, JWTs — need a **waypoint**: a standalone Envoy for L7. You opt in at the **namespace** level (one waypoint for every service in `petshop`, the common default) or per-service. We reset the L4 policies here so the L7 story stands on its own (in production you'd keep both — defence in depth).


In [ ]:
# reset the L4 policies for a clean L7 story
kubectl --context $CTX -n petshop delete authorizationpolicy allow-storefront allow-checkout --ignore-not-found

# a waypoint (istio-waypoint GatewayClass) …
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: petstore-waypoint
  namespace: petshop
  labels: { istio.io/waypoint-for: service }
spec:
  gatewayClassName: istio-waypoint
  listeners:
  - name: mesh
    port: 15008
    protocol: HBONE
EOF
# … then enrol the whole petshop NAMESPACE to use it (one waypoint for every service)
kubectl --context $CTX label namespace petshop istio.io/use-waypoint=petstore-waypoint --overwrite
kubectl --context $CTX -n petshop rollout status deploy/petstore-waypoint --timeout=150s


**Namespace or per-service?** We enrolled the whole namespace, so this one waypoint serves every service in `petshop` — the common default: fewer proxies, and you keep full policy granularity because the `AuthorizationPolicy` below still targets the waypoint/service. Switch to **per-service** (`kubectl label service petstore … istio.io/use-waypoint=petstore-waypoint`) only when you want to isolate a sensitive service, scale its waypoint independently, or keep its L7 path (and any waypoint restart) separate from the rest of the namespace.


## 10 · An IdP, and a real JWT

Keycloak runs in its own (non-ambient) namespace with a `petshop` realm and two users: **alice** (`role: user`) and **bob** (`role: admin`). We **started it back in §1** so it's already warm — here we just confirm it, then mint a token and read its claims (the issuer and `realm_access.roles` are what the waypoint checks).


In [ ]:
# mint + decode a token (run from a petshop pod; it reaches Keycloak cross-namespace)
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
tok() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=$1 -d password=$1 -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p'; }
BOB=$(tok bob)
echo "$BOB" | cut -d. -f2 | { read p; python3 - "$p" <<'PY'
import sys,base64,json
p=sys.argv[1]; p+="="*(-len(p)%4)
d=json.loads(base64.urlsafe_b64decode(p))
print("iss:  ", d["iss"]); print("user: ", d["preferred_username"]); print("roles:", d["realm_access"]["roles"])
PY
}


## 11 · Authorize on the JWT, at L7

Two objects on the waypoint. `RequestAuthentication` validates the token against Keycloak's JWKS. `AuthorizationPolicy` then requires a valid token for **any** request and — with a CEL `when` clause over the nested `realm_access.roles` claim — restricts `DELETE` to admins.


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: RequestAuthentication
metadata: { name: petshop-jwt, namespace: petshop }
spec:
  targetRefs:
  - { kind: Gateway, group: gateway.networking.k8s.io, name: petstore-waypoint }
  jwtRules:
  - issuer:  "http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop"
    jwksUri: "http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/certs"
    forwardOriginalToken: true
---
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: petstore-jwt-authz, namespace: petshop }
spec:
  targetRefs:
  - { kind: Gateway, group: gateway.networking.k8s.io, name: petstore-waypoint }
  action: ALLOW
  rules:
  - from: [{ source: { requestPrincipals: ["*"] } }]           # any valid token may read
    to:   [{ operation: { methods: ["GET"] } }]
  - from: [{ source: { requestPrincipals: ["*"] } }]           # only admins may delete
    to:   [{ operation: { methods: ["DELETE"] } }]
    when: [{ key: "request.auth.claims[realm_access][roles]", values: ["admin"] }]
EOF
sleep 10


In [ ]:
# the matrix: no token, alice (user), bob (admin)
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
tok() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=$1 -d password=$1 -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p'; }
call() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c "$1" ; }
ALICE=$(tok alice); BOB=$(tok bob)
echo "no token    GET  /pets     -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 http://petstore:8080/pets")"
echo "alice       GET  /pets     -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets")"
echo "alice(user) DELETE /pets/1 -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -X DELETE -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets/1")"
echo "bob(admin)  DELETE /pets/1 -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -X DELETE -H 'Authorization: Bearer $BOB' http://petstore:8080/pets/1")"


## 12 · What Solo Enterprise adds

We have added several Enterprise capabilities already: the Solo distribution mesh, the **Solo UI** from §2 — and §13 adds the per-workload cert claims next.

In [ ]:
# L7 telemetry config on ztunnel. On the 1.30 line the Solo build reports it
# directly: metrics, access-log and tracing enabled, pointing at an OTel
# collector endpoint. The enriched HTTP fields (method/path/response_code)
# flow through that licensed Solo telemetry pipeline — the identity-aware L4
# access logs in §6 are what ztunnel always writes to stdout.
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=cert-identity-worker -o jsonpath='{.items[0].metadata.name}')
istioctl --context $CTX ztunnel-config all "$ZT.$ISTIO_NS" -o json | jq '.config.l7Config'


## 13 · Closing the gap — workload claims

§7 hit a wall: two pods on one ServiceAccount are indistinguishable at L4 — one SVID between them. **Workload claims** close it: with `ENABLE_WORKLOAD_CLAIMS=true`, ztunnel requests a certificate **per pod**, istiod embeds claims in it at issuance, and the `AuthorizationPolicy` matches them with CEL — still at L4, still no waypoint. The mesh is already on the 1.30 line, so this is one Helm value on ztunnel.

We left the flag off until now on purpose — the shared-cert gap in §4/§7 **is** the story this step closes. And workload claims is a pure **L4** capability, so the L7 waypoint and JWT policies from §9–§11 come off first: the claims story reads cleanest with only L4 policy applied.


In [ ]:
# back to a pure L4 story: remove the waypoint + JWT policies from §9-§11
kubectl --context $CTX -n petshop delete authorizationpolicy petstore-jwt-authz --ignore-not-found
kubectl --context $CTX -n petshop delete requestauthentication petshop-jwt --ignore-not-found
kubectl --context $CTX label namespace petshop istio.io/use-waypoint-
kubectl --context $CTX -n petshop delete gateway petstore-waypoint --ignore-not-found


In [ ]:
# ONE new value: ENABLE_WORKLOAD_CLAIMS on ztunnel. Same chart, same version.
helm --kube-context $CTX upgrade -i ztunnel $HREPO/ztunnel -n $ISTIO_NS --version $HVER --wait -f - <<EOF
profile: ambient
hub: $HUB
tag: $TAG
namespace: $ISTIO_NS
istioNamespace: $ISTIO_NS
env:
  LOG_FORMAT: json
  L7_ENABLED: "true"
  ENABLE_WORKLOAD_CLAIMS: "true"    # per-POD certs + claim enforcement
EOF
kubectl --context $CTX -n $ISTIO_NS rollout status daemonset/ztunnel --timeout=180s


**The flag flips certificates mesh-wide.** ztunnel now keys its certificate cache by **pod**, not ServiceAccount — compare §4, where checkout-blue and checkout-green resolved to one shared SVID. Every leaf below names a pod, and `WIT present` says the cert carries a workload identity token with that pod's claims.


In [ ]:
# every workload now holds a PER-POD cert (contrast §4: one per ServiceAccount)
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=cert-identity-worker -o jsonpath='{.items[0].metadata.name}')
istioctl --context $CTX ztunnel-config certificates "$ZT.$ISTIO_NS" | grep -E "CERTIFICATE|checkout"


**Annotate the pod — the claim is embedded in its certificate at issuance.** istiod reads `solo.io.security-claims/<key>` off the pod and bakes it into the cert, alongside auto claims for the workload name, namespace and pod. The SPIFFE URI never changes (still `sa/checkout`); the claims ride alongside it.


In [ ]:
# blue is the gold tier, green is silver — one annotation each, pods roll
kubectl --context $CTX -n petshop patch deploy checkout-blue  -p '{"spec":{"template":{"metadata":{"annotations":{"solo.io.security-claims/tier":"gold"}}}}}'
kubectl --context $CTX -n petshop patch deploy checkout-green -p '{"spec":{"template":{"metadata":{"annotations":{"solo.io.security-claims/tier":"silver"}}}}}'
kubectl --context $CTX -n petshop rollout status deploy/checkout-blue deploy/checkout-green --timeout=120s
sleep 5


In [ ]:
# open the certs with openssl and read the claims off the SAN. Each cert gains an
# otherName SAN under Solo's OID (1.3.6.1.4.1.65865.1.1) whose payload is base64url JSON.
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=cert-identity-worker -o jsonpath='{.items[0].metadata.name}')
istioctl --context $CTX ztunnel-config certificates "$ZT.$ISTIO_NS" -o json > /tmp/zt-certs.json

for pod in checkout-blue checkout-green; do
  # the leaf cert ztunnel presents for this pod (the JSON wraps the PEM in base64)
  jq -r ".[] | select(.identity | contains(\"$pod\")) | .certChain[0].pem" /tmp/zt-certs.json \
    | base64 -d > /tmp/$pod.pem

  echo "── $pod ──────────────────────────────────────────────"
  openssl x509 -in /tmp/$pod.pem -noout -text | grep -A1 "Subject Alternative Name" | tail -1

  # decode the otherName payload -> the claims JSON (base64url, no padding)
  openssl x509 -in /tmp/$pod.pem -noout -text | grep -o '65865\.1\.1:.*' | cut -d: -f2 \
    | python3 -c 'import sys,base64,json; p=sys.stdin.read().strip(); print(json.dumps(json.loads(base64.urlsafe_b64decode(p+"="*(-len(p)%4))), indent=2))'
done


Same URI SAN on both (`…/sa/checkout`) — but blue's cert says `tier: gold` and green's says `tier: silver`, signed by istiod. Now authorize on it: the same `when` CEL shape as §8, over `source.claims` instead of `source.namespace` (the `/` in the annotation key becomes `.` in the claim key). Fail-closed: a workload with no claim never matches the ALLOW, so grant with ALLOW on annotated workloads rather than DENY on a missing claim.


In [ ]:
# allow ONLY the gold tier at L4 — enforced by ztunnel, per hop, no waypoint
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata:
  name: allow-gold-checkout
  namespace: petshop
spec:
  selector:
    matchLabels: { app: petstore }
  action: ALLOW
  rules:
    - from:
        - source:
            principals:
              - cert-identity/ns/petshop/sa/checkout
      to:
        - operation:
            ports: ["8080"]
      when:
        - key: "source.claims['solo.io.security-claims.tier']"
          values: ["gold"]
EOF
kubectl --context $CTX -n petshop get authorizationpolicy allow-gold-checkout -o jsonpath='{.status.conditions[0].message}'; echo; echo

# fresh per-pod certs + the new policy take ~20-30s to converge — wait for both
# outcomes to settle (blue allowed AND green denied)
for i in $(seq 1 24); do
  B=$(kubectl --context $CTX -n petshop exec deploy/checkout-blue -- sh -c \
        'curl -s -o /dev/null -w "%{http_code}" --max-time 3 http://petstore:8080/' 2>/dev/null)
  G=$(kubectl --context $CTX -n petshop exec deploy/checkout-green -- sh -c \
        'curl -s -o /dev/null -w "%{http_code}" --max-time 3 http://petstore:8080/' 2>/dev/null)
  [ "$B" = "200" ] && [ "$G" != "200" ] && break; sleep 5
done

for pod in checkout-blue checkout-green; do
  kubectl --context $CTX -n petshop exec deploy/$pod -- sh -c \
    "curl -s -o /dev/null -w '$pod -> %{http_code}\n' --max-time 5 http://petstore:8080/ || echo '$pod -> DENIED at L4'"
done


`checkout-blue -> 200`, `checkout-green -> DENIED` — same ServiceAccount, told apart at L4 by the workload's own certificate. That is the exact thing §7 could not do. The policy status reads `attached to ztunnel` — ztunnel itself enforces this, per hop. Note the distinction from §11: **this** CEL is over claims in the workload *certificate*; the JWT CEL in §11 is over the user's *request token* at the waypoint. The auto claims (`istio.io.workload.pod` and friends) are matchable the same way. The policy also lives in `yaml/60-claims/10-allow-gold-checkout.yaml`.


In [ ]:
# tidy up
kind delete cluster --name cert-identity
